# MDC_51006_EP - Foundations of Machine Learning - Lab 3

The goals of this lab are:
- to understand how to optimize hyperparameters
- to play with automl

We will reuse what we have done last week. We are going to use the same dataset as well as the final metric.

In [ ]:
!uv pip install -U seaborn polars scikit-learn

Using Python 3.12.11 environment at: /usr
Resolved 20 packages in 115ms
Audited 20 packages in 0.73ms


In [ ]:
!rm german_credit_data_biased_training.csv
!wget https://raw.githubusercontent.com/IBM/monitor-wml-model-with-watson-openscale/master/data/german_credit_data_biased_training.csv

rm: cannot remove 'german_credit_data_biased_training.csv': No such file or directory
--2025-10-01 07:24:20--  https://raw.githubusercontent.com/IBM/monitor-wml-model-with-watson-openscale/master/data/german_credit_data_biased_training.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 689622 (673K) [text/plain]
Saving to: ‘german_credit_data_biased_training.csv’

german_credit_data_ 100%[===================>] 673.46K  --.-KB/s    in 0.01s   

2025-10-01 07:24:20 (66.6 MB/s) - ‘german_credit_data_biased_training.csv’ saved [689622/689622]



In [ ]:
import polars as pl

In [ ]:
german_credit = pl.read_csv("german_credit_data_biased_training.csv")

In [ ]:
import seaborn.objects as so

In [ ]:
import polars.selectors as cs
string_columns = german_credit.select(cs.string()).columns
numerical_columns = german_credit.select(cs.numeric()).columns

In [ ]:
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
categorical_features = string_columns.copy()
categorical_features.remove('Risk')
categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
numerical_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())
    ])

preprocessor = ColumnTransformer(
        transformers=[
            ('cat', categorical_transformer, categorical_features),
            ('num', numerical_transformer, numerical_columns)
        ],
        remainder='passthrough'
    )


In [ ]:
real_prop = {'Risk': .02, 'No Risk': .98}
train_prop = {'Risk': 1/3, 'No Risk': 2/3}

In [ ]:
custom_weight = {'Risk': real_prop['Risk']/train_prop['Risk'], 'No Risk': real_prop['No Risk']/train_prop['No Risk']}

In [ ]:
def compute_costs(LoanAmount):
  return({'Risk_No Risk': 5.0 + .6 * LoanAmount, 'No Risk_No Risk': 1.0 - .05 * LoanAmount,
          'Risk_Risk': 1.0, 'No Risk_Risk': 1.0})

In [ ]:
def custom_metric(y_true, y_pred, LoanAmount):
  costs = compute_costs(LoanAmount)
  y_pred = pl.Series(y_pred)
  loss = (y_true=='Risk') * custom_weight['Risk'] *\
                ((y_pred=='Risk') * costs['Risk_Risk'] + (y_pred=='No Risk') * costs['Risk_No Risk']) +\
         (y_true=='No Risk') * custom_weight['No Risk'] *\
                ((y_pred=='Risk') * costs['No Risk_Risk'] + (y_pred=='No Risk') * costs['No Risk_No Risk'])
  return(loss.mean())

In [ ]:
sklearn.set_config(enable_metadata_routing=True)
custom_scorer = sklearn.metrics.make_scorer(custom_metric, greater_is_better=False).set_score_request(LoanAmount=True)

In [ ]:
import numpy as np
def custom_predict_from_proba(proba, LoanAmount):
  classes_ = np.array(['No Risk', 'Risk'])
  costs = compute_costs(LoanAmount)
  decision = np.vstack([( proba[:, 0] * custom_weight['No Risk'] * costs['No Risk_No Risk'] + proba[:, 1] * custom_weight['Risk'] * costs['Risk_No Risk']),
                       ( proba[:, 0] * custom_weight['No Risk'] * costs['No Risk_Risk'] + proba[:, 1] * custom_weight['Risk'] * costs['Risk_Risk'])])
  return(pl.Series(classes_[decision.argmin(axis=0)]))

def custom_predict_from_single_proba(proba, LoanAmount):
  proba=np.column_stack((1.0-proba, proba))
  return(custom_predict_from_proba(proba, LoanAmount))

def custom_predict(pipeline, X_test, LoanAmount):
  proba = pipeline.predict_proba(X_test)
  return(custom_predict_from_proba(proba, LoanAmount))

In [ ]:
def custom_metric_from_proba(y_true, proba, LoanAmount):
  y_pred = custom_predict_from_single_proba(proba, LoanAmount)
  return(custom_metric(y_true, y_pred, LoanAmount))

custom_scorer_from_proba = sklearn.metrics.make_scorer(custom_metric_from_proba, response_method="predict_proba", greater_is_better=False).set_score_request(LoanAmount=True, sample_weights=True)

In [ ]:
def compute_weights(y_true, LoanAmount):
  costs = compute_costs(LoanAmount)
  return(
      (y_true=='Risk') * custom_weight['Risk'] * (costs['Risk_No Risk']-costs['Risk_Risk'])
      + (y_true=='No Risk') * custom_weight['No Risk'] * (costs['No Risk_Risk']-costs['No Risk_No Risk'])
  )

In [ ]:
X = german_credit.drop('Risk')
y = german_credit['Risk']
LoanAmount = X['LoanAmount']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test, LoanAmount_train, LoanAmount_test = train_test_split(X, y, LoanAmount, test_size=0.2, random_state=42)

In [ ]:
from sklearn.model_selection import cross_val_score

## Histogram Gradient Boosting

Today, the first method, we consider is the Histogram Gradient Boosting from Scikit Learn. We can obtain quickly a first classifier using

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

In [ ]:
categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
numerical_transformer = Pipeline(steps=[
        ('scaler', StandardScaler().set_fit_request(sample_weight=True))
    ])

preprocessor = ColumnTransformer(
        transformers=[
            ('cat', categorical_transformer, categorical_features),
            ('num', numerical_transformer, numerical_columns)
        ],
        remainder='passthrough'
    )
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', HistGradientBoostingClassifier().set_fit_request(sample_weight=True))]).set_output(transform="polars")

**---**

In [ ]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


We can look at its performance on the test set

In [ ]:
custom_scorer(pipeline, X_test, y_test, LoanAmount=LoanAmount_test)

86.415699

and verify that this is coherent with the result obtained by cross validation

In [ ]:
cross_val_score(pipeline, X_train, y_train, scoring=custom_scorer, params = {'LoanAmount': LoanAmount_train})

array([83.30090813, 85.86226125, 90.60388688, 92.23252125, 84.58267875])

**Question** Verify that with the same predictor, one can obtain a better score using the Bayes classifier of the last lab.

**Answer** It is just a matter of using the probability as we have done last week to "tweak" the predictions:

In [ ]:
custom_scorer_from_proba(pipeline, X_test, y_test, LoanAmount=LoanAmount_test)

94.78372649999999

In [ ]:
cross_val_score(pipeline, X_train, y_train, scoring=custom_scorer_from_proba, params = {'LoanAmount': LoanAmount_train})

array([90.91325625, 90.78874875, 97.56331125, 95.04454875, 90.96742312])

---

**Question** Can you obtain a better performance using the 'sample_weight' approach?

**Answer**

In [ ]:
from sklearn.base import clone
pipeline_weighted = clone(pipeline)

In [ ]:
pipeline_weighted.fit(X_train, y_train, sample_weight=compute_weights(y_train, LoanAmount_train))

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
custom_scorer(pipeline_weighted, X_test, y_test, LoanAmount=LoanAmount_test)

95.126679

---

## Hyparameter Search with Grid Search

Most methods, including Histogram Gradient Boosting, have default parameters. They are not necessarily the best ones. The easiest way to obtain a good set of hyperparameters is to define a list of set of hyperparameters and to iterate cross validation on this list. The most natural way to construct this list is to make a grid over possible values over each hyperparameter. Note that this does not scale well with the number of hyperparameters to optimize!

Scikit Learn provides an easy way to perform such an optimization (exploration?).



In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__learning_rate": (0.0001, 0.001, .01, 0.1),
    "classifier__max_leaf_nodes": (2, 3, 5),
}
pipeline_grid_search = GridSearchCV(pipeline, param_grid=param_grid, n_jobs=2, cv=5)
pipeline_grid_search.fit(X_train, y_train)

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'classifier__learning_rate': (0.0001, ...), 'classifier__max_leaf_nodes': (2, ...)}"
,scoring,None
,n_jobs,2
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('cat', ...), ('num', ...)]"


In [ ]:
custom_scorer(pipeline_grid_search, X_test, y_test, LoanAmount=LoanAmount_test)

89.21894850000001

In [ ]:
custom_scorer_from_proba(pipeline_grid_search, X_test, y_test, LoanAmount=LoanAmount_test)

96.43945650000002

**Question** Use the `scoring` argument of GridSearchCV to use our `custom_scorer_from_proba` to select the best hyperparameter set.

**Answer**

In [ ]:
pipeline_grid_search = GridSearchCV(pipeline, param_grid=param_grid, n_jobs=2, cv=5, scoring=custom_scorer_from_proba)
pipeline_grid_search.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1135: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


,estimator,Pipeline(step...lassifier())])
,param_grid,"{'classifier__learning_rate': (0.0001, ...), 'classifier__max_leaf_nodes': (2, ...)}"
,scoring,make_scorer(c...redict_proba')
,n_jobs,2
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('cat', ...), ('num', ...)]"


In [ ]:
custom_scorer_from_proba(pipeline_grid_search, X_test, y_test, LoanAmount=LoanAmount_test)

69.98594849999999

---

**Question** Can you use the weights in the optimization ?

**Answer** It seems that this is not (yet?) possible!

---

In [ ]:
pipeline.set_score_request(sample_weight=True)
pipeline_grid_search = GridSearchCV(pipeline, param_grid=param_grid, n_jobs=2, cv=5)
pipeline_grid_search.fit(X_train, y_train, sample_weight=compute_weights(y_train, LoanAmount_train))

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1135: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


,estimator,Pipeline(step...lassifier())])
,param_grid,"{'classifier__learning_rate': (0.0001, ...), 'classifier__max_leaf_nodes': (2, ...)}"
,scoring,None
,n_jobs,2
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('cat', ...), ('num', ...)]"


In [ ]:
custom_scorer(pipeline_grid_search, X_test, y_test, LoanAmount=LoanAmount_test)

69.98594849999999

---

## Optuna

Optuna is a generic optimization library. It allows to optimize in a *clever* way a set of mixed (continuous and discrete) parameters. It is also very versatile: the only requirement is to be able to compute a score for each set of parameters!

We need to install it:

In [ ]:
!uv pip install -U optuna

Using Python 3.12.11 environment at: /usr
Resolved 12 packages in 132ms
Prepared 4 packages in 58ms
Uninstalled 2 packages in 5ms
Installed 4 packages in 9ms
 + colorlog==6.9.0
 - markupsafe==3.0.2
 + markupsafe==3.0.3
 + optuna==4.5.0
 - pyyaml==6.0.2
 + pyyaml==6.0.3


In [ ]:
import optuna

We define then a function that will generate a model from a set of parameters

In [ ]:
def get_pipeline(classifier_params={}):
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', HistGradientBoostingClassifier(**classifier_params).set_fit_request(sample_weight=True))]).set_output(transform="polars")

    return pipeline



and we can optimize this set easily with the following code structure:

In [ ]:
def objective(trial):
    classifier_params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 1, log=True),
        'max_depth': trial.suggest_int('max_leaf_nodes', 2, 10),
        'l2_regularization': trial.suggest_float('l2_regularization', 0, 10)
    }

    pipeline = get_pipeline(classifier_params)
    pipeline.fit(X_train, y_train)

    score = custom_scorer_from_proba(pipeline, X_test, y_test, LoanAmount=LoanAmount_test)

    trial.set_user_attr('score', score)

    return score

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2025-10-01 07:25:19,309] A new study created in memory with name: no-name-3e3cffeb-e94b-4f35-8138-4c5fcee3f95d
[I 2025-10-01 07:25:19,720] Trial 0 finished with value: 93.29949899999998 and parameters: {'learning_rate': 0.28002629774021326, 'max_leaf_nodes': 5, 'l2_regularization': 3.3847205944200454}. Best is trial 0 with value: 93.29949899999998.
[I 2025-10-01 07:25:20,387] Trial 1 finished with value: 96.90082949999999 and parameters: {'learning_rate': 0.02319735865517377, 'max_leaf_nodes': 7, 'l2_regularization': 1.1919807159002405}. Best is trial 1 with value: 96.90082949999999.
[I 2025-10-01 07:25:20,611] Trial 2 finished with value: 94.9729395 and parameters: {'learning_rate': 0.5507935832232368, 'max_leaf_nodes': 2, 'l2_regularization': 2.355249667537689}. Best is trial 1 with value: 96.90082949999999.
[I 2025-10-01 07:25:21,024] Trial 3 finished with value: 98.70024000000001 and parameters: {'learning_rate': 0.11103721380720037, 'max_leaf_nodes': 5, 'l2_regularization': 9.7

We can obtain the best parameters with

In [ ]:
study.best_params

{'learning_rate': 0.04553797954501644,
 'max_leaf_nodes': 7,
 'l2_regularization': 7.327539478494691}

In [ ]:
pipeline = get_pipeline(study.best_params)

In [ ]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
custom_scorer_from_proba(pipeline, X_test, y_test, LoanAmount=LoanAmount_test)

96.10258800000001

We can also visualize the evolution of the score:

In [ ]:
from optuna.visualization import plot_optimization_history

plot_optimization_history(**{
    'study': study
})



**Question** Modify the previous code to rely on cross validation rather than the use of the test set

**Answer**

In [ ]:
def objective(trial):
    classifier_params = {
        'learning_rate': trial.suggest_float('learning_rate', 0, 1),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 2, 10),
        'l2_regularization': trial.suggest_float('l2_regularization', 0, 10),
        'random_state': 37
    }

    pipeline = get_pipeline(classifier_params)

    gain = cross_val_score(pipeline, X_train, y_train, scoring=custom_scorer_from_proba, params = {'LoanAmount': LoanAmount_train}).mean()

    trial.set_user_attr('gain', gain)

    return gain

In [ ]:
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=37),
                            pruner=optuna.pruners.MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=100)

[I 2025-10-01 07:26:29,565] A new study created in memory with name: no-name-e7f6aba4-2d7b-457d-8746-56777ec883c0
[I 2025-10-01 07:26:30,981] Trial 0 finished with value: 89.67858937500002 and parameters: {'learning_rate': 0.9444966028573069, 'max_leaf_nodes': 6, 'l2_regularization': 1.927950289571576}. Best is trial 0 with value: 89.67858937500002.
[I 2025-10-01 07:26:32,244] Trial 1 finished with value: 92.7277545 and parameters: {'learning_rate': 0.581894876879992, 'max_leaf_nodes': 7, 'l2_regularization': 6.8422401746317405}. Best is trial 1 with value: 92.7277545.
[I 2025-10-01 07:26:33,646] Trial 2 finished with value: 95.10734625 and parameters: {'learning_rate': 0.10343755501999807, 'max_leaf_nodes': 8, 'l2_regularization': 2.8197907007423595}. Best is trial 2 with value: 95.10734625.
[I 2025-10-01 07:26:35,060] Trial 3 finished with value: 88.818183375 and parameters: {'learning_rate': 0.7534245871442832, 'max_leaf_nodes': 9, 'l2_regularization': 6.274005962793598}. Best is tr

In [ ]:
pipeline = get_pipeline(study.best_params)

In [ ]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
custom_scorer_from_proba(pipeline, X_test, y_test, LoanAmount=LoanAmount_test)

96.2302575

---

**Question** What about the use of weights?

**Answer**

In [ ]:
def objective(trial):
    classifier_params = {
        'learning_rate': trial.suggest_float('learning_rate', 0, 1),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 2, 10),
        'l2_regularization': trial.suggest_float('l2_regularization', 0, 10),
        'random_state': 37
    }

    pipeline = get_pipeline(classifier_params)

    gain = cross_val_score(pipeline, X_train, y_train, scoring=custom_scorer, params = {'LoanAmount': LoanAmount_train, 'sample_weight': compute_weights(y_train, LoanAmount_train)}).mean()

    trial.set_user_attr('gain', gain)

    return gain

In [ ]:
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=37),
                            pruner=optuna.pruners.MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=100)

[I 2025-10-01 07:28:17,041] A new study created in memory with name: no-name-eb397973-b1ca-4f86-be5e-f71858284684
[I 2025-10-01 07:28:18,231] Trial 0 finished with value: 82.19207212500001 and parameters: {'learning_rate': 0.9444966028573069, 'max_leaf_nodes': 6, 'l2_regularization': 1.927950289571576}. Best is trial 0 with value: 82.19207212500001.
[I 2025-10-01 07:28:19,487] Trial 1 finished with value: 86.31988162500001 and parameters: {'learning_rate': 0.581894876879992, 'max_leaf_nodes': 7, 'l2_regularization': 6.8422401746317405}. Best is trial 1 with value: 86.31988162500001.
[I 2025-10-01 07:28:20,876] Trial 2 finished with value: 94.89018712499998 and parameters: {'learning_rate': 0.10343755501999807, 'max_leaf_nodes': 8, 'l2_regularization': 2.8197907007423595}. Best is trial 2 with value: 94.89018712499998.
[I 2025-10-01 07:28:22,800] Trial 3 finished with value: 80.5127895 and parameters: {'learning_rate': 0.7534245871442832, 'max_leaf_nodes': 9, 'l2_regularization': 6.2740

In [ ]:
pipeline = get_pipeline(study.best_params)

In [ ]:
pipeline.fit(X_train, y_train, sample_weight=compute_weights(y_train, LoanAmount_train))

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
custom_scorer(pipeline, X_test, y_test, LoanAmount=LoanAmount_test)

94.336782

Note that without our custom scorer, we could have used the integration with Scikit Learn:

In [ ]:
!uv pip install optuna-integration

Using Python 3.12.11 environment at: /usr
Resolved 13 packages in 45ms
Prepared 1 package in 16ms
Installed 1 package in 3ms
 + optuna-integration==4.5.0


In [ ]:
from optuna_integration import OptunaSearchCV
optuna_parameters = {
    "classifier__learning_rate": optuna.distributions.FloatDistribution(0.01, 0.1, log=True),
    "classifier__max_depth": optuna.distributions.IntDistribution(2,10),
    "classifier__l2_regularization": optuna.distributions.FloatDistribution(0, 10)
        }

Boost_pipeline_Optuna_CV_clf = OptunaSearchCV(pipeline, optuna_parameters, cv=10, n_jobs=-1, n_trials=10)


/tmp/ipython-input-2233814130.py:8: ExperimentalWarning:

OptunaSearchCV is experimental (supported from v0.17.0). The interface can change in the future.



In [ ]:
Boost_pipeline_Optuna_CV_clf.fit(X_train, y_train)

[I 2025-10-01 07:29:59,919] A new study created in memory with name: no-name-93c55290-c1c7-494f-afe3-ffb651551243
[I 2025-10-01 07:30:03,305] Trial 1 finished with value: 0.7849999999999999 and parameters: {'classifier__learning_rate': 0.08116082555451673, 'classifier__max_depth': 3, 'classifier__l2_regularization': 0.745497336012998}. Best is trial 1 with value: 0.7849999999999999.
[I 2025-10-01 07:30:03,409] Trial 0 finished with value: 0.776 and parameters: {'classifier__learning_rate': 0.02807361265076538, 'classifier__max_depth': 2, 'classifier__l2_regularization': 1.2127032300423457}. Best is trial 1 with value: 0.7849999999999999.
[I 2025-10-01 07:30:06,460] Trial 2 finished with value: 0.77625 and parameters: {'classifier__learning_rate': 0.028096879396879246, 'classifier__max_depth': 10, 'classifier__l2_regularization': 4.106328024776513}. Best is trial 1 with value: 0.7849999999999999.
[I 2025-10-01 07:30:06,475] Trial 3 finished with value: 0.7815000000000001 and parameters:

,estimator,Pipeline(step...af_nodes=2))])
,param_distributions,"{'classifier__l2_regularization': FloatDistribu....0, step=None), 'classifier__learning_rate': FloatDistribu...01, step=None), 'classifier__max_depth': IntDistributi...low=2, step=1)}"
,cv,10
,enable_pruning,False
,error_score,nan
,max_iter,1000
,n_jobs,-1
,n_trials,10
,random_state,None
,refit,True
,return_train_score,False


---

## XGBoost

Histogram Gradient Boosting is quite similar to the XGBoost algorithm, but they are not equivalent! We can try to see which one is the best for our dataset.


In [ ]:
from xgboost import XGBClassifier

In [ ]:
pipeline_xgb = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', XGBClassifier(objective="binary:logistic").set_fit_request(sample_weight=True))])

In [ ]:
try:
  pipeline_xgb.fit(X_train, y_train, sample_weight=compute_weights(y_train, LoanAmount_train))
except Exception as e:
    print(e)

Invalid classes inferred from unique values of `y`.  Expected: [0 1], got ['No Risk' 'Risk']



A limitation of the XGBoost implementation is that the target labels are expected to be integer. As this is not the case here, we need to encode those labels...

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(y_train)

LabelEncoder()

In [ ]:
pipeline_xgb.fit(X_train, le.transform(y_train), sample_weight=compute_weights(y_train, LoanAmount_train))

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
custom_predict_from_single_proba(pipeline_xgb.predict_proba(X_test), LoanAmount_test)

""
str
"""Risk"""
"""Risk"""
"""Risk"""
"""No Risk"""
"""No Risk"""
…
"""Risk"""
"""Risk"""
"""Risk"""


In [ ]:
custom_scorer_from_proba(pipeline_xgb, X_test, y_test, LoanAmount=LoanAmount_test)

88.20186149999999

**Question** Optimize the hyperparameters for this model

**Answer**

In [ ]:
def custom_metric_from_proba_and_inverse_transform(y_true, proba, LoanAmount):
  y_pred = custom_predict_from_single_proba(proba, LoanAmount)
  return(custom_metric(le.inverse_transform(y_true), y_pred, LoanAmount))

custom_scorer_from_proba_and_inverse_transform = sklearn.metrics.make_scorer(custom_metric_from_proba_and_inverse_transform, response_method="predict_proba", greater_is_better=False).set_score_request(LoanAmount=True, sample_weights=True)

In [ ]:
def objective(trial):
    classifier_params = {
        "verbosity": 0,
        "objective": "binary:logistic",
        # use exact for small dataset.
        "tree_method": "exact",
        # defines booster, gblinear for linear functions.
        "booster": trial.suggest_categorical("booster", ["gbtree", "gblinear", "dart"]),
        # L2 regularization weight.
        "lambda": trial.suggest_float("lambda", 1e-8, 1.0, log=True),
        # L1 regularization weight.
        "alpha": trial.suggest_float("alpha", 1e-8, 1.0, log=True),
        # sampling ratio for training data.
        "subsample": trial.suggest_float("subsample", 0.2, 1.0),
        # sampling according to each tree.
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0),
    }

    if classifier_params["booster"] in ["gbtree", "dart"]:
        # maximum depth of the tree, signifies complexity of the tree.
        classifier_params["max_depth"] = trial.suggest_int("max_depth", 3, 9, step=2)
        # minimum child weight, larger the term more conservative the tree.
        classifier_params["min_child_weight"] = trial.suggest_int("min_child_weight", 2, 10)
        classifier_params["eta"] = trial.suggest_float("eta", 1e-8, 1.0, log=True)
        # defines how selective algorithm is.
        classifier_params["gamma"] = trial.suggest_float("gamma", 1e-8, 1.0, log=True)
        classifier_params["grow_policy"] = trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"])

    if classifier_params["booster"] == "dart":
        classifier_params["sample_type"] = trial.suggest_categorical("sample_type", ["uniform", "weighted"])
        classifier_params["normalize_type"] = trial.suggest_categorical("normalize_type", ["tree", "forest"])
        classifier_params["rate_drop"] = trial.suggest_float("rate_drop", 1e-8, 1.0, log=True)
        classifier_params["skip_drop"] = trial.suggest_float("skip_drop", 1e-8, 1.0, log=True)

    pipeline = get_pipeline(classifier_params)

    gain = cross_val_score(pipeline, X_train, le.transform(y_train), scoring=custom_scorer_from_proba_and_inverse_transform, params = {'LoanAmount': LoanAmount_train}).mean()
    trial.set_user_attr('gain', gain)

    return gain

In [ ]:
def get_pipeline(classifier_params={}):
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', XGBClassifier(**classifier_params).set_fit_request(sample_weight=True))]).set_output(transform="polars")

    return pipeline

In [ ]:
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=37),
                            pruner=optuna.pruners.MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=100)

[I 2025-10-01 07:30:16,500] A new study created in memory with name: no-name-ec70050c-5511-4e6d-8e0d-003883953827
[I 2025-10-01 07:30:19,818] Trial 0 finished with value: 81.38466337499999 and parameters: {'booster': 'gbtree', 'lambda': 0.0004520214197603329, 'alpha': 0.0009134266563560387, 'subsample': 0.7473792139705393, 'colsample_bytree': 0.28275004401599846, 'max_depth': 7, 'min_child_weight': 4, 'eta': 0.010651154813385614, 'gamma': 0.021960642221781016, 'grow_policy': 'depthwise'}. Best is trial 0 with value: 81.38466337499999.
[I 2025-10-01 07:30:22,298] Trial 1 finished with value: 69.47791875 and parameters: {'booster': 'gbtree', 'lambda': 0.0005856107838120437, 'alpha': 0.0003863808235682347, 'subsample': 0.7639594833426497, 'colsample_bytree': 0.9434096147206608, 'max_depth': 3, 'min_child_weight': 3, 'eta': 8.634318185854628e-08, 'gamma': 8.27557322106628e-06, 'grow_policy': 'lossguide'}. Best is trial 0 with value: 81.38466337499999.
[I 2025-10-01 07:30:23,294] Trial 2 fi

In [ ]:
study.best_params

{'booster': 'dart',
 'lambda': 2.178613831878639e-05,
 'alpha': 3.3580850915379916e-06,
 'subsample': 0.8803634778917188,
 'colsample_bytree': 0.7083855791488248,
 'max_depth': 3,
 'min_child_weight': 8,
 'eta': 0.13584064544397859,
 'gamma': 2.0725646231560108e-05,
 'grow_policy': 'lossguide',
 'sample_type': 'weighted',
 'normalize_type': 'forest',
 'rate_drop': 7.04943728420483e-06,
 'skip_drop': 0.5281446098477587}

In [ ]:
pipeline = get_pipeline(study.best_params)

In [ ]:
pipeline.fit(X_train, le.transform(y_train))

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
custom_scorer_from_proba(pipeline, X_test, y_test, LoanAmount=LoanAmount_test)

95.39237100000001

---

## LightGBM

**Question:** What about LightGBM ?

**Answer:**

In [ ]:
!uv pip install -U lightgbm

Using Python 3.12.11 environment at: /usr
Resolved 3 packages in 33ms
Audited 3 packages in 0.27ms


In [ ]:
from lightgbm import LGBMClassifier
from sklearn.preprocessing import OrdinalEncoder, FunctionTransformer

In [ ]:
def convert_to_category(x):
   return(x.astype('category'))

categorical_transformer_lgbm = Pipeline(
    steps=[
        ("encoder", OrdinalEncoder(dtype=np.int32)),
        ("converter", FunctionTransformer(convert_to_category, check_inverse=False)),
    ]
)

In [ ]:
preprocessor_lgbm = ColumnTransformer(
        transformers=[
            ('cat', categorical_transformer_lgbm, categorical_features),
            ('num', numerical_transformer, numerical_columns)
        ],
        remainder='passthrough'
    )

In [ ]:
pipeline_lgbm = Pipeline(steps=[('preprocessor', preprocessor_lgbm),
                              ('classifier', LGBMClassifier(objective="binary"))])
pipeline_lgbm.set_output(transform="pandas")

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
pipeline_lgbm.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 1335, number of negative: 2665
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000917 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 449
[LightGBM] [Info] Number of data points in the train set: 4000, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333750 -> initscore=-0.691273
[LightGBM] [Info] Start training from score -0.691273


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
custom_scorer_from_proba(pipeline_lgbm, X_test, y_test, LoanAmount=LoanAmount_test)

94.2568095

---

## Automl

To conclude this part, we may use an automl framework and see whether we can do better than our tuned boosting models.

`flaml`is an interesting possibility as it is more or less compatible with `scikit learn`.

**Question** Can you obtain better performances with it?

**Answer**

In [ ]:
!uv pip install -U flaml pyspark

Using Python 3.12.11 environment at: /usr
Resolved 4 packages in 3.61s
Prepared 3 packages in 37.69s
Uninstalled 2 packages in 77ms
Installed 3 packages in 28ms
 + flaml==2.3.6
 - py4j==0.10.9.7
 + py4j==0.10.9.9
 - pyspark==3.5.1
 + pyspark==4.0.1


In [ ]:
from flaml import AutoML

In [ ]:
from sklearn.base import is_classifier
is_classifier(AutoML(task="classification"))

False

In [ ]:
preprocessor = ColumnTransformer(
        transformers=[
            ('cat', categorical_transformer, categorical_features),
            ('num', numerical_transformer, numerical_columns)
        ],
        remainder='passthrough'
    )

In [ ]:
pipeline_automl = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', AutoML(task='classification', time_budget=60, metric='log_loss'))])

In [ ]:
import pandas as pd
X_train_pd = X_train.to_pandas()
y_train_pd = y_train.to_pandas()

In [ ]:
pipeline_automl.fit(X_train_pd, y_train_pd)

[flaml.automl.logger: 10-01 07:38:06] {1752} INFO - task = classification
[flaml.automl.logger: 10-01 07:38:06] {1763} INFO - Evaluation method: holdout
[flaml.automl.logger: 10-01 07:38:06] {1862} INFO - Minimizing error metric: log_loss
[flaml.automl.logger: 10-01 07:38:06] {1979} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 10-01 07:38:06] {2282} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 10-01 07:38:06] {2417} INFO - Estimated sufficient time budget=484s. Estimated necessary time budget=11s.
[flaml.automl.logger: 10-01 07:38:06] {2466} INFO -  at 0.1s,	estimator lgbm's best error=0.5687,	best estimator lgbm's best error=0.5687
[flaml.automl.logger: 10-01 07:38:06] {2282} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 10-01 07:38:06] {2466} INFO -  at 0.1s,	estimator lgbm's best error=0.5687,	best estimator lgbm's best error=0.5687
[flaml.automl.logger: 10-

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 10-01 07:38:06] {2466} INFO -  at 0.4s,	estimator sgd's best error=0.4562,	best estimator sgd's best error=0.4562
[flaml.automl.logger: 10-01 07:38:06] {2282} INFO - iteration 4, current learner xgboost
[flaml.automl.logger: 10-01 07:38:07] {2466} INFO -  at 0.5s,	estimator xgboost's best error=0.5687,	best estimator sgd's best error=0.4562
[flaml.automl.logger: 10-01 07:38:07] {2282} INFO - iteration 5, current learner extra_tree
[flaml.automl.logger: 10-01 07:38:07] {2466} INFO -  at 0.6s,	estimator extra_tree's best error=0.5040,	best estimator sgd's best error=0.4562
[flaml.automl.logger: 10-01 07:38:07] {2282} INFO - iteration 6, current learner lgbm
[flaml.automl.logger: 10-01 07:38:07] {2466} INFO -  at 0.6s,	estimator lgbm's best error=0.4402,	best estimator lgbm's best error=0.4402
[flaml.automl.logger: 10-01 07:38:07] {2282} INFO - iteration 7, current learner lgbm
[flaml.automl.logger: 10-01 07:38:07] {2466} INFO -  at 0.7s,	estimator lgbm's best error=

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names



[flaml.automl.logger: 10-01 07:38:07] {2466} INFO -  at 0.9s,	estimator xgboost's best error=0.5687,	best estimator lgbm's best error=0.4384
[flaml.automl.logger: 10-01 07:38:07] {2282} INFO - iteration 12, current learner xgboost
[flaml.automl.logger: 10-01 07:38:07] {2466} INFO -  at 1.0s,	estimator xgboost's best error=0.5075,	best estimator lgbm's best error=0.4384
[flaml.automl.logger: 10-01 07:38:07] {2282} INFO - iteration 13, current learner extra_tree
[flaml.automl.logger: 10-01 07:38:07] {2466} INFO -  at 1.0s,	estimator extra_tree's best error=0.4731,	best estimator lgbm's best error=0.4384
[flaml.automl.logger: 10-01 07:38:07] {2282} INFO - iteration 14, current learner lgbm
[flaml.automl.logger: 10-01 07:38:07] {2466} INFO -  at 1.1s,	estimator lgbm's best error=0.4327,	best estimator lgbm's best error=0.4327
[flaml.automl.logger: 10-01 07:38:07] {2282} INFO - iteration 15, current learner rf
[flaml.automl.logger: 10-01 07:38:07] {2466} INFO -  at 1.2s,	estimator rf's best

...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names



[flaml.automl.logger: 10-01 07:38:42] {2466} INFO -  at 35.6s,	estimator xgb_limitdepth's best error=0.4140,	best estimator xgboost's best error=0.4115
[flaml.automl.logger: 10-01 07:38:42] {2282} INFO - iteration 453, current learner xgb_limitdepth
[flaml.automl.logger: 10-01 07:38:42] {2466} INFO -  at 35.7s,	estimator xgb_limitdepth's best error=0.4140,	best estimator xgboost's best error=0.4115
[flaml.automl.logger: 10-01 07:38:42] {2282} INFO - iteration 454, current learner xgboost
[flaml.automl.logger: 10-01 07:38:42] {2466} INFO -  at 35.8s,	estimator xgboost's best error=0.4115,	best estimator xgboost's best error=0.4115
[flaml.automl.logger: 10-01 07:38:42] {2282} INFO - iteration 455, current learner xgboost
[flaml.automl.logger: 10-01 07:38:42] {2466} INFO -  at 35.8s,	estimator xgboost's best error=0.4115,	best estimator xgboost's best error=0.4115
[flaml.automl.logger: 10-01 07:38:42] {2282} INFO - iteration 456, current learner xgboost
[flaml.automl.logger: 10-01 07:38:4

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 10-01 07:39:00] {2466} INFO -  at 53.8s,	estimator lrl1's best error=0.4308,	best estimator xgb_limitdepth's best error=0.4106
[flaml.automl.logger: 10-01 07:39:00] {2282} INFO - iteration 575, current learner lrl1
[flaml.automl.logger: 10-01 07:39:00] {2466} INFO -  at 54.0s,	estimator lrl1's best error=0.4295,	best estimator xgb_limitdepth's best error=0.4106
[flaml.automl.logger: 10-01 07:39:00] {2282} INFO - iteration 576, current learner rf
[flaml.automl.logger: 10-01 07:39:00] {2466} INFO -  at 54.3s,	estimator rf's best error=0.4211,	best estimator xgb_limitdepth's best error=0.4106
[flaml.automl.logger: 10-01 07:39:00] {2282} INFO - iteration 577, current learner xgb_limitdepth
[flaml.automl.logger: 10-01 07:39:00] {2466} INFO -  at 54.3s,	estimator xgb_limitdepth's best error=0.4106,	best estimator xgb_limitdepth's best error=0.4106
[flaml.automl.logger: 10-01 07:39:00] {2282} INFO - iteration 578, current learner lrl1
[flaml.automl.logger: 10-01 07:39:00

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
X_test_pd = X_test.to_pandas()
y_test_pd = y_test.to_numpy()
LoanAmount_test_pd = LoanAmount_test.to_numpy()

In [ ]:
pipeline_automl.predict_proba(X_test_pd)

array([[0.5540693 , 0.44593072],
       [0.97125846, 0.02874155],
       [0.9357144 , 0.06428557],
       ...,
       [0.96243   , 0.03757003],
       [0.31617576, 0.68382424],
       [0.9476924 , 0.05230759]], shape=(1000, 2), dtype=float32)

In [ ]:
custom_metric_from_proba(y_test, pipeline_automl.predict_proba(X_test_pd)[:,1], LoanAmount_test)

-95.1847815

In [ ]:
try:
  custom_scorer_from_proba(pipeline_automl, X_test_pd, y_test_pd, LoanAmount=LoanAmount_test_pd)
except Exception as e:
    print(e)

Pipeline should either be a classifier to be used with response_method=predict_proba or the response_method should be 'predict'. Got a regressor with response_method=predict_proba instead.


---